# 06 — Model Training & Evaluation

**Project:** Early Disease Risk Predictor  
**Phase:** 4 — Training & Evaluation  

---

This notebook trains all four model architectures defined in Phase 3 and rigorously evaluates them.

| Section | Content |
|---|---|
| 1 | Load preprocessed data |
| 2 | Stratified 70 / 15 / 15 train–val–test split |
| 3 | 5-fold cross-validation on the training set |
| 4 | Train M1 Logistic Regression |
| 5 | Train M2 Random Forest |
| 6 | Train M3 XGBoost via `RandomizedSearchCV` |
| 7 | Train M4 MLP Neural Network |
| 8 | Model comparison on validation set |
| 9 | Metric analysis — why Precision / Recall / F1 / AUC-ROC |
| 10 | Final test-set evaluation |
| 11 | Error analysis — false-negative and false-positive profiles |
| 12 | Bias check — subgroup performance by age, BMI, and source |
| 13 | Model persistence |

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    RandomizedSearchCV, cross_validate
)
from sklearn.linear_model        import LogisticRegression
from sklearn.ensemble             import RandomForestClassifier
from sklearn.neural_network       import MLPClassifier
from sklearn.utils.class_weight   import compute_sample_weight
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    accuracy_score, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from xgboost     import XGBClassifier
from scipy.stats import uniform, randint

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

PROC_DIR  = os.path.join('..', 'data', 'processed')
FIGS_DIR  = os.path.join('..', 'reports', 'figures')
MODEL_DIR = os.path.join('..', 'data', 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

DISEASES     = ['diabetes', 'heart_disease', 'hypertension']
RANDOM_STATE = 42
CV_FOLDS     = 5

# Load configs from Phase 3
with open(os.path.join(PROC_DIR, 'model_configs.json')) as fh:
    cfg = json.load(fh)
SCALE_POS_WEIGHTS = cfg['scale_pos_weight']

print('Setup complete.')
print('scale_pos_weight:', SCALE_POS_WEIGHTS)

## 1. Load Preprocessed Data

In [ ]:
datasets = {}
for disease in DISEASES:
    X = pd.read_csv(os.path.join(PROC_DIR, f'X_{disease}.csv'))
    y = pd.read_csv(os.path.join(PROC_DIR, f'y_{disease}.csv')).squeeze()
    datasets[disease] = (X, y)
    print(f'{disease:15s} | X{X.shape}  pos_rate={y.mean():.3f}  '
          f'(n_pos={int(y.sum())}  n_neg={int((y==0).sum())})')

FEATURE_NAMES = list(datasets['diabetes'][0].columns)
print('\nFeatures:', FEATURE_NAMES)

## 2. Stratified 70 / 15 / 15 Split

**Rationale:** A three-way split provides independent sets for:
- **Train (70 %)** — model fitting and hyperparameter search via cross-validation  
- **Validation (15 %)** — XGBoost early-stopping monitor; final model selection  
- **Test (15 %)** — held-out set touched only once for the final reported metrics  

`stratify=y` at both split steps preserves the positive class ratio in every partition.

In [ ]:
splits = {}
print(f'{'Disease':15s}  {'Train':>6}  {'Val':>5}  {'Test':>5}  '
      f'{'pos_tr':>7}  {'pos_va':>7}  {'pos_te':>7}')
print('-' * 65)

for disease, (X, y) in datasets.items():
    # Step 1: 70% train, 30% temp
    X_tr, X_tmp, y_tr, y_tmp = train_test_split(
        X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
    )
    # Step 2: split the 30% evenly → 15% val + 15% test
    X_va, X_te, y_va, y_te = train_test_split(
        X_tmp, y_tmp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_tmp
    )
    splits[disease] = (X_tr, X_va, X_te, y_tr, y_va, y_te)
    print(f'{disease:15s}  {len(X_tr):>6}  {len(X_va):>5}  {len(X_te):>5}  '
          f'{y_tr.mean():>7.3f}  {y_va.mean():>7.3f}  {y_te.mean():>7.3f}')

In [ ]:
# Visualise split sizes and class balance
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel 1: sample counts per partition
split_counts = {d: [len(splits[d][i]) for i in range(3)] for d in DISEASES}
df_counts = pd.DataFrame(split_counts, index=['Train', 'Val', 'Test']).T
df_counts.plot(kind='bar', ax=axes[0], color=['#4C9BE8', '#FFA040', '#E85D5D'],
               edgecolor='white', width=0.65)
axes[0].set_title('Sample Counts per Partition')
axes[0].set_ylabel('Samples')
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend()

# Panel 2: positive rate per partition
pos_rates = {
    d: [splits[d][3].mean(), splits[d][4].mean(), splits[d][5].mean()]
    for d in DISEASES
}
df_pos = pd.DataFrame(pos_rates, index=['Train', 'Val', 'Test']).T
df_pos.plot(kind='bar', ax=axes[1], color=['#4C9BE8', '#FFA040', '#E85D5D'],
            edgecolor='white', width=0.65)
axes[1].set_title('Positive Rate per Partition (stratification check)')
axes[1].set_ylabel('Positive rate')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend()

plt.suptitle('Train / Val / Test Split (70 / 15 / 15)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR, '15_split_overview.png'))
plt.show()

## 3. Evaluation Helpers

In [ ]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)


def evaluate(model, X, y, label=''):
    """Return a dict of metrics; optionally print a one-liner."""
    y_pred  = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]
    m = {
        'Accuracy' : round(accuracy_score(y, y_pred), 4),
        'AUC-ROC'  : round(roc_auc_score(y, y_proba), 4),
        'Precision': round(precision_score(y, y_pred, zero_division=0), 4),
        'Recall'   : round(recall_score(y, y_pred, zero_division=0), 4),
        'F1'       : round(f1_score(y, y_pred, zero_division=0), 4),
    }
    if label:
        print(f'  {label:<8} AUC={m["AUC-ROC"]:.4f}  '
              f'F1={m["F1"]:.4f}  Prec={m["Precision"]:.4f}  Rec={m["Recall"]:.4f}')
    return m


def cv_auc_with_weights(clf_class, params, X, y, cv):
    """5-fold CV with balanced sample weights computed per fold."""
    aucs = []
    for tr_idx, va_idx in cv.split(X, y):
        X_f_tr, y_f_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_f_va, y_f_va = X.iloc[va_idx],  y.iloc[va_idx]
        sw = compute_sample_weight('balanced', y_f_tr)
        clf = clf_class(**params)
        clf.fit(X_f_tr, y_f_tr, sample_weight=sw)
        prob = clf.predict_proba(X_f_va)[:, 1]
        aucs.append(roc_auc_score(y_f_va, prob))
    return np.array(aucs)


print('Helpers ready.  CV:', cv)

## 4. M1 — Logistic Regression (Baseline)

In [ ]:
LR_PARAMS = dict(
    class_weight='balanced', max_iter=1000,
    solver='lbfgs', C=1.0, random_state=RANDOM_STATE
)

lr_models    = {}
lr_cv_scores = {}

print('=== M1: Logistic Regression ===')
for disease, (X_tr, X_va, X_te, y_tr, y_va, y_te) in splits.items():
    clf = LogisticRegression(**LR_PARAMS)
    cv_res = cross_validate(clf, X_tr, y_tr, cv=cv,
                             scoring='roc_auc', return_train_score=False)
    lr_cv_scores[disease] = cv_res['test_score']
    clf.fit(X_tr, y_tr)
    lr_models[disease] = clf
    print(f'\n[{disease.replace("_", " ").title()}]')
    print(f'  CV AUC : {cv_res["test_score"].mean():.4f} '
          f'± {cv_res["test_score"].std():.4f}')
    evaluate(clf, X_va, y_va, 'Val ')

print('\nLogistic Regression — done.')

## 5. M2 — Random Forest

In [ ]:
RF_PARAMS = dict(
    n_estimators=300, max_depth=10, min_samples_leaf=5,
    max_features='sqrt', class_weight='balanced',
    n_jobs=-1, random_state=RANDOM_STATE
)

rf_models    = {}
rf_cv_scores = {}

print('=== M2: Random Forest ===')
for disease, (X_tr, X_va, X_te, y_tr, y_va, y_te) in splits.items():
    clf = RandomForestClassifier(**RF_PARAMS)
    cv_res = cross_validate(clf, X_tr, y_tr, cv=cv,
                             scoring='roc_auc', return_train_score=False)
    rf_cv_scores[disease] = cv_res['test_score']
    clf.fit(X_tr, y_tr)
    rf_models[disease] = clf
    print(f'\n[{disease.replace("_", " ").title()}]')
    print(f'  CV AUC : {cv_res["test_score"].mean():.4f} '
          f'± {cv_res["test_score"].std():.4f}')
    evaluate(clf, X_va, y_va, 'Val ')

print('\nRandom Forest — done.')

## 6. M3 — XGBoost with RandomizedSearchCV

**Protocol:**
1. `RandomizedSearchCV` samples 40 configurations from the search space, evaluated with 5-fold stratified CV on the **training set** only (scored by `roc_auc`).
2. The best configuration is used to build the final model, which is trained on the training set with **early stopping** monitored on the validation set (`early_stopping_rounds=20`).
3. The test set remains untouched until Section 10.

In [ ]:
XGB_PARAM_DIST = {
    'learning_rate'    : uniform(0.01, 0.29),
    'n_estimators'     : randint(100, 501),
    'max_depth'        : randint(3, 7),
    'min_child_weight' : randint(1, 6),
    'subsample'        : uniform(0.60, 0.40),
    'colsample_bytree' : uniform(0.60, 0.40),
    'gamma'            : uniform(0.00, 0.30),
    'reg_alpha'        : uniform(0.00, 0.50),
    'reg_lambda'       : uniform(0.50, 1.50),
}

xgb_models      = {}
xgb_best_params = {}
xgb_cv_scores   = {}

print('=== M3: XGBoost — RandomizedSearchCV (n_iter=40, cv=5) ===')
for disease, (X_tr, X_va, X_te, y_tr, y_va, y_te) in splits.items():
    spw = SCALE_POS_WEIGHTS[disease]

    # ── Step 1: hyperparameter search ────────────────────────────────────────
    base = XGBClassifier(
        objective='binary:logistic', eval_metric='logloss',
        scale_pos_weight=spw, n_jobs=-1, random_state=RANDOM_STATE
    )
    search = RandomizedSearchCV(
        base, XGB_PARAM_DIST,
        n_iter=40, scoring='roc_auc', cv=cv,
        n_jobs=-1, random_state=RANDOM_STATE, verbose=0
    )
    search.fit(X_tr, y_tr)
    xgb_best_params[disease] = search.best_params_
    xgb_cv_scores[disease]   = search.best_score_

    # ── Step 2: final fit with early stopping on val set ─────────────────────
    clf = XGBClassifier(
        **search.best_params_,
        objective='binary:logistic', eval_metric='logloss',
        scale_pos_weight=spw,
        early_stopping_rounds=20,
        n_jobs=-1, random_state=RANDOM_STATE
    )
    clf.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    xgb_models[disease] = clf

    print(f'\n[{disease.replace("_", " ").title()}]')
    print(f'  Best CV AUC  : {search.best_score_:.4f}')
    print(f'  Trees used   : {clf.best_iteration + 1} '
          f'(from {search.best_params_["n_estimators"]} searched)')
    evaluate(clf, X_va, y_va, 'Val ')

print('\nXGBoost — done.')

In [ ]:
# ── Best hyperparameter summary table ────────────────────────────────────────
param_order = [
    'learning_rate', 'n_estimators', 'max_depth', 'min_child_weight',
    'subsample', 'colsample_bytree', 'gamma', 'reg_alpha', 'reg_lambda'
]

rows = []
for p in param_order:
    rows.append({
        'Parameter'   : p,
        'Diabetes'    : round(xgb_best_params['diabetes'][p], 4),
        'Heart Dis.'  : round(xgb_best_params['heart_disease'][p], 4),
        'Hypertension': round(xgb_best_params['hypertension'][p], 4),
    })

df_params = pd.DataFrame(rows).set_index('Parameter')
print('Best XGBoost Hyperparameters per Disease')
print('=' * 55)
print(df_params.to_string())

# save for Phase 5
joblib.dump(xgb_best_params, os.path.join(PROC_DIR, 'xgb_best_params.pkl'))
print('\nBest params saved.')

## 7. M4 — MLP Neural Network

Cross-validation for the MLP uses a **manual fold loop** so that `sample_weight` can be recomputed from each fold's training labels — this avoids the information-leak that would occur if weights were computed on the full training set before splitting.

In [ ]:
MLP_PARAMS = dict(
    hidden_layer_sizes=(128, 64), activation='relu', solver='adam',
    alpha=0.001, batch_size=64, learning_rate='adaptive',
    learning_rate_init=0.001, max_iter=500,
    early_stopping=True, validation_fraction=0.10,
    n_iter_no_change=20, random_state=RANDOM_STATE
)

mlp_models    = {}
mlp_cv_scores = {}

print('=== M4: MLP Neural Network ===')
for disease, (X_tr, X_va, X_te, y_tr, y_va, y_te) in splits.items():
    # 5-fold CV with per-fold sample weights
    cv_aucs = cv_auc_with_weights(MLPClassifier, MLP_PARAMS, X_tr, y_tr, cv)
    mlp_cv_scores[disease] = cv_aucs

    # Final fit on full training set
    sw = compute_sample_weight('balanced', y_tr)
    clf = MLPClassifier(**MLP_PARAMS)
    clf.fit(X_tr, y_tr, sample_weight=sw)
    mlp_models[disease] = clf

    print(f'\n[{disease.replace("_", " ").title()}]')
    print(f'  CV AUC  : {cv_aucs.mean():.4f} ± {cv_aucs.std():.4f}')
    print(f'  Epochs  : {clf.n_iter_}')
    evaluate(clf, X_va, y_va, 'Val ')

print('\nMLP — done.')

In [ ]:
# ── MLP training loss curves ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, disease in zip(axes, DISEASES):
    clf = mlp_models[disease]
    ax.plot(clf.loss_curve_, color='#4C9BE8', linewidth=1.6)
    ax.axvline(clf.n_iter_ - 1, color='#E85D5D', linestyle='--',
               linewidth=1, label=f'Stop @ epoch {clf.n_iter_}')
    ax.set_title(disease.replace('_', ' ').title())
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Log Loss')
    ax.legend(fontsize=8)
plt.suptitle('MLP Training Loss Curves (log loss vs epoch)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR, '16_mlp_loss_curves.png'))
plt.show()

## 8. Model Comparison — Validation Set

In [ ]:
ALL_MODELS = {
    'Logistic Reg.'  : lr_models,
    'Random Forest'  : rf_models,
    'XGBoost (Tuned)': xgb_models,
    'MLP Neural Net' : mlp_models,
}

val_rows = []
for model_name, model_dict in ALL_MODELS.items():
    for disease in DISEASES:
        X_tr, X_va, X_te, y_tr, y_va, y_te = splits[disease]
        m = evaluate(model_dict[disease], X_va, y_va)
        val_rows.append({'Model': model_name, 'Disease': disease, **m})

df_val = pd.DataFrame(val_rows)

# AUC-ROC pivot table
pivot = df_val.pivot(index='Model', columns='Disease', values='AUC-ROC')
pivot.columns = [c.replace('_', ' ').title() for c in pivot.columns]
print('Validation AUC-ROC')
print(pivot.round(4).to_string())

In [ ]:
# ── CV AUC scores: mean ± std per model per disease ───────────────────────────
cv_score_store = {
    'Logistic Reg.'  : lr_cv_scores,
    'Random Forest'  : rf_cv_scores,
    'XGBoost (Tuned)': {d: np.full(CV_FOLDS, xgb_cv_scores[d]) for d in DISEASES},
    'MLP Neural Net' : mlp_cv_scores,
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
colors = ['#888888', '#4C9BE8', '#E85D5D', '#FFA040']

for ax, disease in zip(axes, DISEASES):
    means = [cv_score_store[m][disease].mean() for m in ALL_MODELS]
    stds  = [cv_score_store[m][disease].std()  for m in ALL_MODELS]
    bars  = ax.bar(list(ALL_MODELS.keys()), means, yerr=stds,
                   color=colors, edgecolor='white', capsize=5, width=0.6)
    ax.bar_label(bars, fmt='%.3f', padding=4, fontsize=8)
    ax.set_title(disease.replace('_', ' ').title())
    ax.set_ylabel('CV AUC-ROC')
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=18)

plt.suptitle('5-Fold CV AUC-ROC per Model  (mean ± std)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR, '17_cv_auc_comparison.png'))
plt.show()

## 9. Metric Analysis — Justification

### Why not accuracy?

Diabetes positive rate is 5 %. A classifier that always predicts **Negative** achieves **95 % accuracy** while catching zero actual patients — useless for screening.

### Metrics used and why

| Metric | Formula | Why it matters here |
|---|---|---|
| **AUC-ROC** | Area under TPR vs FPR curve | Threshold-free; robust under class imbalance; primary tuning target |
| **Recall (Sensitivity)** | TP / (TP + FN) | A missed diagnosis (FN) is far more dangerous than a false alarm → maximise Recall |
| **Precision** | TP / (TP + FP) | High false-positive rate causes unnecessary follow-ups and patient anxiety → track Precision |
| **F1-Score** | 2 · (Prec · Rec) / (Prec + Rec) | Harmonic mean; penalises extreme imbalances between Precision and Recall |
| **Accuracy** | (TP+TN) / N | Reported for completeness; misleading for imbalanced targets |

### Medical priority order

In a screening context the cost hierarchy is:
$$\text{cost}(\text{FN}) \gg \text{cost}(\text{FP}) \gg \text{cost}(\text{TN})$$

Therefore the **primary clinical metric is Recall**. AUC-ROC is used for tuning because it is threshold-independent; the operating threshold can be lowered in deployment to trade Precision for Recall.

## 10. Final Test-Set Evaluation

The test set is used **exactly once**. All four models are evaluated; XGBoost (tuned) is the primary system.

In [ ]:
test_rows = []
print('=== Final Test-Set Evaluation ===')
for model_name, model_dict in ALL_MODELS.items():
    print(f'\n--- {model_name} ---')
    for disease in DISEASES:
        X_tr, X_va, X_te, y_tr, y_va, y_te = splits[disease]
        m = evaluate(model_dict[disease], X_te, y_te,
                     disease.replace('_', ' ').title())
        test_rows.append({'Model': model_name, 'Disease': disease, **m})

df_test = pd.DataFrame(test_rows)

In [ ]:
# Full classification report for the primary model (XGBoost)
print('=== XGBoost — Classification Report (Test Set) ===')
for disease in DISEASES:
    X_tr, X_va, X_te, y_tr, y_va, y_te = splits[disease]
    y_pred = xgb_models[disease].predict(X_te)
    print(f'\n[{disease.replace("_", " ").title()}]')
    print(classification_report(y_te, y_pred,
                                 target_names=['Negative', 'Positive'],
                                 digits=4))

In [ ]:
# ── ROC curves — test set ─────────────────────────────────────────────────────
roc_colors = {
    'Logistic Reg.'  : '#888888',
    'Random Forest'  : '#4C9BE8',
    'XGBoost (Tuned)': '#E85D5D',
    'MLP Neural Net' : '#FFA040',
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, disease in zip(axes, DISEASES):
    X_tr, X_va, X_te, y_tr, y_va, y_te = splits[disease]
    for model_name, model_dict in ALL_MODELS.items():
        y_prob = model_dict[disease].predict_proba(X_te)[:, 1]
        auc = roc_auc_score(y_te, y_prob)
        RocCurveDisplay.from_predictions(
            y_te, y_prob,
            name=f'{model_name} ({auc:.3f})',
            ax=ax, color=roc_colors[model_name]
        )
    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Random (0.500)')
    ax.set_title(disease.replace('_', ' ').title())
    ax.legend(fontsize=7, loc='lower right')

plt.suptitle('ROC Curves — Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR, '18_roc_curves_test.png'))
plt.show()

In [ ]:
# ── Confusion matrices — XGBoost primary model ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, disease in zip(axes, DISEASES):
    X_tr, X_va, X_te, y_tr, y_va, y_te = splits[disease]
    ConfusionMatrixDisplay.from_estimator(
        xgb_models[disease], X_te, y_te,
        display_labels=['Negative', 'Positive'],
        colorbar=False, ax=ax, cmap='Blues'
    )
    ax.set_title(f'XGBoost — {disease.replace("_", " ").title()}')

plt.suptitle('Confusion Matrices — XGBoost (Test Set)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR, '19_confusion_matrices.png'))
plt.show()

## 11. Error Analysis

We examine **False Negatives (FN)** — sick patients the model missed — and **False Positives (FP)** — healthy patients incorrectly flagged.  
By comparing the mean standardised feature values of FN, FP, and True Positive (TP) groups we identify which clinical profiles the model struggles with most.

In [ ]:
error_summary = {}

for disease in DISEASES:
    X_tr, X_va, X_te, y_tr, y_va, y_te = splits[disease]
    clf = xgb_models[disease]

    y_pred  = pd.Series(clf.predict(X_te),         index=X_te.index)
    y_proba = pd.Series(clf.predict_proba(X_te)[:,1], index=X_te.index)
    y_true  = y_te.reset_index(drop=False).set_index(y_te.index).iloc[:, -1]

    fn_mask = (y_te == 1) & (y_pred == 0)   # missed diagnoses
    fp_mask = (y_te == 0) & (y_pred == 1)   # false alarms
    tp_mask = (y_te == 1) & (y_pred == 1)   # correctly identified

    n_pos = int(y_te.sum())
    n_neg = int((y_te == 0).sum())
    fn_n  = int(fn_mask.sum())
    fp_n  = int(fp_mask.sum())

    print(f'\n=== {disease.replace("_", " ").title()} ===')
    print(f'  Positives in test  : {n_pos}')
    print(f'  False Negatives    : {fn_n}  '
          f'({fn_n/n_pos*100:.1f}% of positives missed)')
    print(f'  False Positives    : {fp_n}  '
          f'({fp_n/n_neg*100:.1f}% of negatives flagged)')

    if fn_n > 0 and tp_mask.sum() > 0:
        profile = pd.DataFrame({
            'FN — missed' : X_te[fn_mask].mean(),
            'TP — caught' : X_te[tp_mask].mean(),
            'FP — alarm'  : X_te[fp_mask].mean() if fp_n > 0 else np.nan,
        }).round(3)
        print(profile.to_string())
    error_summary[disease] = {
        'fn_rate': fn_n / n_pos if n_pos else 0,
        'fp_rate': fp_n / n_neg if n_neg else 0,
    }

In [ ]:
# ── FN / FP rate bar chart ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(DISEASES))
w = 0.35
fn_rates = [error_summary[d]['fn_rate'] * 100 for d in DISEASES]
fp_rates = [error_summary[d]['fp_rate'] * 100 for d in DISEASES]

b1 = ax.bar(x - w/2, fn_rates, w, label='FN rate (% of positives missed)',
             color='#E85D5D', edgecolor='white')
b2 = ax.bar(x + w/2, fp_rates, w, label='FP rate (% of negatives flagged)',
             color='#FFA040', edgecolor='white')
ax.bar_label(b1, fmt='%.1f%%', padding=3, fontsize=9)
ax.bar_label(b2, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels([d.replace('_', ' ').title() for d in DISEASES])
ax.set_ylabel('Error rate (%)')
ax.set_title('XGBoost — False Negative & False Positive Rates (Test Set)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGS_DIR, '20_fn_fp_rates.png'))
plt.show()

## 12. Bias Check — Subgroup Performance

A model may achieve high average AUC while performing poorly on a specific demographic or clinical subgroup — this is a form of algorithmic bias.  
We evaluate **Recall** (sensitivity) separately for subgroups defined by:

| Grouping | Subgroups | Source |
|---|---|---|
| Age | Young (<40), Middle (40–55), Older (>55) | `master.csv` raw age |
| BMI | Normal (<25), Overweight (25–30), Obese (>30) | `master.csv` raw BMI |
| Dataset source | PIMA, Framingham, UCI Heart | `source` column |

Recall is the primary bias metric: a low-recall subgroup means the model systematically misses patients from that group.

In [ ]:
master = pd.read_csv(os.path.join(PROC_DIR, 'master.csv'))

bias_rows = []

for disease in DISEASES:
    X_tr, X_va, X_te, y_tr, y_va, y_te = splits[disease]
    clf = xgb_models[disease]

    y_pred  = clf.predict(X_te)
    y_proba = clf.predict_proba(X_te)[:, 1]

    # Attach raw features from master using index alignment
    raw = master.loc[X_te.index].copy()
    raw['y_true'] = y_te.values
    raw['y_pred'] = y_pred
    raw['y_prob'] = y_proba

    # Define subgroups
    raw['age_group'] = pd.cut(
        raw['age'],
        bins=[0, 40, 55, 200],
        labels=['Young (<40)', 'Middle (40-55)', 'Older (>55)']
    )
    raw['bmi_group'] = pd.cut(
        raw['bmi'],
        bins=[0, 25, 30, 200],
        labels=['Normal (<25)', 'Overweight (25-30)', 'Obese (>30)']
    )
    raw['source_group'] = raw['source']

    for grp_col, grp_label in [
        ('age_group',    'Age'),
        ('bmi_group',    'BMI'),
        ('source_group', 'Source'),
    ]:
        for grp_val, grp_df in raw.groupby(grp_col, observed=True):
            n_pos = int(grp_df['y_true'].sum())
            if n_pos < 2:
                continue
            try:
                auc  = roc_auc_score(grp_df['y_true'], grp_df['y_prob'])
            except Exception:
                auc  = float('nan')
            rec  = recall_score(grp_df['y_true'], grp_df['y_pred'], zero_division=0)
            prec = precision_score(grp_df['y_true'], grp_df['y_pred'], zero_division=0)
            f1   = f1_score(grp_df['y_true'], grp_df['y_pred'], zero_division=0)
            bias_rows.append({
                'Disease'   : disease.replace('_', ' ').title(),
                'Group type': grp_label,
                'Subgroup'  : str(grp_val),
                'n'         : len(grp_df),
                'n_pos'     : n_pos,
                'AUC-ROC'   : round(auc, 3),
                'Recall'    : round(rec, 3),
                'Precision' : round(prec, 3),
                'F1'        : round(f1, 3),
            })

df_bias = pd.DataFrame(bias_rows)
print(df_bias.to_string(index=False))

In [ ]:
# ── Recall heatmap by subgroup × disease ─────────────────────────────────────
for grp_label in ['Age', 'BMI', 'Source']:
    sub = df_bias[df_bias['Group type'] == grp_label].copy()
    if sub.empty:
        continue

    pivot = sub.pivot_table(
        index='Subgroup', columns='Disease', values='Recall'
    )

    fig, ax = plt.subplots(figsize=(9, max(3, len(pivot) * 0.9)))
    sns.heatmap(
        pivot, annot=True, fmt='.3f', cmap='RdYlGn',
        vmin=0, vmax=1, linewidths=0.5, ax=ax,
        cbar_kws={'label': 'Recall'}
    )
    ax.set_title(f'Recall by {grp_label} Subgroup  (XGBoost · Test Set)',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')
    plt.tight_layout()
    fname = f'21_bias_{grp_label.lower()}_recall.png'
    plt.savefig(os.path.join(FIGS_DIR, fname))
    plt.show()
    print(f'Saved {fname}')

### Bias Interpretation Guide

| Observation | Implication |
|---|---|
| Recall < 0.50 for a subgroup | Model misses more than half the positive cases in that group — potential clinical equity issue |
| Large Recall gap between age groups | Model may rely on age-correlated features; younger patients with atypical profiles are missed |
| Source-level Recall differences | Dataset-specific feature distributions (e.g. PIMA vs Framingham) introduce collection bias |
| BMI-group variation | Obese vs normal-weight patients may have different glucose/BP distributions not captured by a single model |

## 13. Model Persistence

In [ ]:
save_map = {
    'lr'  : lr_models,
    'rf'  : rf_models,
    'xgb' : xgb_models,
    'mlp' : mlp_models,
}

for prefix, model_dict in save_map.items():
    for disease, clf in model_dict.items():
        path = os.path.join(MODEL_DIR, f'{prefix}_{disease}.pkl')
        joblib.dump(clf, path)
        print(f'Saved: {path}')

# Save the test-set results and bias table for the report
df_test.to_csv(os.path.join(PROC_DIR, 'test_results.csv'), index=False)
df_bias.to_csv(os.path.join(PROC_DIR, 'bias_results.csv'), index=False)
print('\nTest results and bias table saved.')

## 14. Phase 4 Summary

| Item | Detail |
|---|---|
| **Split strategy** | Stratified 70 / 15 / 15 (train / val / test) |
| **Cross-validation** | 5-fold `StratifiedKFold`; scored by `roc_auc` |
| **Hyperparameter tuning** | `RandomizedSearchCV` n_iter=40 on training set (XGBoost) |
| **Early stopping** | XGBoost — `early_stopping_rounds=20` monitored on val set |
| **Imbalance handling** | `scale_pos_weight` (XGBoost) · `class_weight='balanced'` (LR, RF) · `sample_weight` (MLP) |
| **Primary metrics** | AUC-ROC (tuning) · Recall (clinical priority) · F1 · Precision |
| **Error analysis** | FN / FP profiles by mean standardised feature value per disease |
| **Bias check** | Recall reported per age group, BMI group, and source dataset |
| **Saved artefacts** | `{prefix}_{disease}.pkl` × 12 models · `test_results.csv` · `bias_results.csv` |
| **Next phase** | Phase 5 — SHAP explainability dashboard |